In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/food_delivery.csv')
print(df.shape)
print("--------------")
print(df.dtypes)

(43862, 20)
--------------
ID                                 str
Delivery_person_ID                 str
Delivery_person_Age            float64
Delivery_person_Ratings        float64
Restaurant_latitude            float64
Restaurant_longitude           float64
Delivery_location_latitude     float64
Delivery_location_longitude    float64
Order_Date                         str
Time_Orderd                        str
Time_Order_picked                  str
Weatherconditions                  str
Road_traffic_density               str
Vehicle_condition                int64
Type_of_order                      str
Type_of_vehicle                    str
multiple_deliveries            float64
Festival                           str
City                               str
Time_taken(min)                float64
dtype: object


In [2]:
print(df.isnull().sum())

ID                             0
Delivery_person_ID             0
Delivery_person_Age            0
Delivery_person_Ratings        0
Restaurant_latitude            0
Restaurant_longitude           0
Delivery_location_latitude     0
Delivery_location_longitude    0
Order_Date                     0
Time_Orderd                    0
Time_Order_picked              0
Weatherconditions              0
Road_traffic_density           0
Vehicle_condition              0
Type_of_order                  0
Type_of_vehicle                0
multiple_deliveries            0
Festival                       0
City                           0
Time_taken(min)                0
dtype: int64


In [9]:
for col in df.select_dtypes(include='str').columns:
    print(col, df[col].unique()[:10])

ID <ArrowStringArray>
['0x4607 ', '0xb379 ', '0x5d6d ', '0x7a6a ', '0x70a2 ', '0x9bb4 ', '0x95b4 ',
 '0x9eb2 ', '0x1102 ', '0xcdcd ']
Length: 10, dtype: str
Delivery_person_ID <ArrowStringArray>
[  'INDORES13DEL02 ',   'BANGRES18DEL02 ',   'BANGRES19DEL01 ',
  'COIMBRES13DEL02 ',   'CHENRES12DEL01 ',    'HYDRES09DEL03 ',
 'RANCHIRES15DEL01 ',    'MYSRES15DEL02 ',    'HYDRES05DEL02 ',
    'DEHRES17DEL01 ']
Length: 10, dtype: str
Order_Date <ArrowStringArray>
['19-03-2022', '25-03-2022', '05-04-2022', '26-03-2022', '11-03-2022',
 '04-03-2022', '14-03-2022', '20-03-2022', '12-02-2022', '13-02-2022']
Length: 10, dtype: str
Time_Orderd <ArrowStringArray>
['11:30:00', '19:45:00', '08:30:00', '18:00:00', '13:30:00', '21:20:00',
 '19:15:00', '17:25:00', '20:55:00', '21:55:00']
Length: 10, dtype: str
Time_Order_picked <ArrowStringArray>
['11:45:00', '19:50:00', '08:45:00', '18:10:00', '13:45:00', '21:30:00',
 '19:30:00', '17:30:00', '21:05:00', '22:10:00']
Length: 10, dtype: str
Weatherconditio

In [4]:
disguised_nulls = df.apply(lambda col: col.astype(str).str.strip().str.contains('NaN').sum())
print(disguised_nulls[disguised_nulls > 0])

Delivery_person_Age        1854
Delivery_person_Ratings    1908
Time_Orderd                1731
Weatherconditions           616
Road_traffic_density        601
multiple_deliveries         993
Festival                    228
City                       1200
dtype: int64


In [5]:
for col in df.select_dtypes(include='str').columns:
    mask = df[col].str.contains('NaN', na=False)
    df.loc[mask, col] = np.nan

In [6]:
# Weatherconditions: "conditions Sunny" -> "Sunny"
df['Weatherconditions'] = df['Weatherconditions'].str.replace('conditions ', '', regex=False)

# Time_taken(min): "(min) 24" -> 24 (int)
df['Time_taken(min)'] = df['Time_taken(min)'].str.replace('(min) ', '', regex=False).astype(float)

# numeric columns still stored as text
df['Delivery_person_Age'] = df['Delivery_person_Age'].astype(float)
df['Delivery_person_Ratings'] = df['Delivery_person_Ratings'].astype(float)
df['multiple_deliveries'] = df['multiple_deliveries'].astype(float)

In [10]:
# numeric imputation
df['Delivery_person_Age'] = df['Delivery_person_Age'].astype(float).fillna(df['Delivery_person_Age'].astype(float).median())
df['Delivery_person_Ratings'] = df['Delivery_person_Ratings'].astype(float).fillna(df['Delivery_person_Ratings'].astype(float).median())

# mode imputation
for col in ['Weatherconditions', 'Road_traffic_density', 'multiple_deliveries', 'Festival', 'City']:
    df[col] = df[col].fillna(df[col].mode()[0])

# drop rows with missing order time
df = df.dropna(subset=['Time_Orderd'])

print(df.isnull().sum().sum())
print(df.shape)

0
(43862, 20)


In [20]:
print(df[['Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude']].describe())

       Restaurant_latitude  Restaurant_longitude  Delivery_location_latitude  \
count         43862.000000          43862.000000                43862.000000   
mean             17.398871             70.764932                   17.462466   
std               7.337197             21.136195                    7.338540   
min               0.000000              0.000000                    0.010000   
25%              12.934179             73.170283                   12.986229   
50%              18.554382             75.898497                   18.633934   
75%              22.732225             78.045359                   22.785049   
max              30.914057             88.433452                   31.054057   

       Delivery_location_longitude  
count                 43862.000000  
mean                     70.828527  
std                      21.136365  
min                       0.010000  
25%                      73.279312  
50%                      75.999490  
75%                 

In [16]:
print(df['Time_taken(min)'].describe())

count    43862.000000
mean        26.293831
std          9.373765
min         10.000000
25%         19.000000
50%         26.000000
75%         32.000000
max         54.000000
Name: Time_taken(min), dtype: float64


In [18]:
Q1 = df['Time_taken(min)'].quantile(0.25)
Q3 = df['Time_taken(min)'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
print(lower, upper)
print((df['Time_taken(min)'] < lower).sum(), (df['Time_taken(min)'] > upper).sum())

-0.5 51.5
0 255


In [21]:
for col in ['Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude']:
    df[col] = df[col].abs()

In [23]:
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)

0
(43862, 20)


In [25]:
mask = (
    (df['Restaurant_latitude'] < 1) | (df['Restaurant_longitude'] < 1) |
    (df['Delivery_location_latitude'] < 1) | (df['Delivery_location_longitude'] < 1)
)
print(mask.sum())
df = df[~mask]
print(df.shape)

3509
(40353, 20)
